# Day 50 — UserProxyAgent & human-in-the-loop

`UserProxyAgent` injects a human into the conversation. In a notebook we supply an
`input_func` so it doesn't block; in production this is where a real person approves.

> Install: `pip install "autogen-agentchat" "autogen-ext[openai]"` and set `OPENAI_API_KEY`. For a local model, pass `base_url=` + `model_info=` to the client.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
assistant = AssistantAgent("assistant", model_client=model_client,
    system_message="Propose ONE idea and ask the user to approve.")
user = UserProxyAgent("user", input_func=lambda prompt: "APPROVE")
team = RoundRobinGroupChat([assistant, user],
                           termination_condition=TextMentionTermination("APPROVE"))

# TODO:
# result = await team.run(task="Suggest a name for a coffee shop.")
# print(result.messages[-1].source, "->", result.messages[-1].content)
# await model_client.close()


## 🔒 Solution

Verified against autogen-agentchat 0.7.5.

In [ ]:
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
assistant = AssistantAgent("assistant", model_client=model_client,
    system_message="Propose ONE idea and ask the user to approve.")
user = UserProxyAgent("user", input_func=lambda prompt: "APPROVE")
team = RoundRobinGroupChat([assistant, user],
                           termination_condition=TextMentionTermination("APPROVE"))

result = await team.run(task="Suggest a name for a coffee shop.")
for m in result.messages:
    print(f"{m.source:10}: {m.content}")
await model_client.close()